In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.stats import spearmanr
from PGAM.GAM_library import general_additive_model
from PGAM.gam_data_handlers import smooths_handler
from utilities.experiment_paths import ExperimentPaths
from utilities.nlx_ntt_read import load_ntt_clusters




In [2]:
paths = ExperimentPaths("SocialPlaceCells")

db = paths.analysis_root / "cellsdb.xlsx"
# open the excel file and read the data into a pandas dataframe
df = pd.read_excel(db)
# extract the relevant columns into numpy arrays
cell_id = df["cell_ID"].values
expname = df["expname"].values
TT = df["TT"].values
cluster = df["cluster"].values




In [5]:
cellidx = 1
cell_TT = df["TT"].values[cellidx]
cell_TT_str = f"{int(cell_TT):02d}"
cell_cluster = df["cluster"].values[cellidx]
cell_expname = df["expname"].values[cellidx]
cell_id_value = df["cell_ID"].values[cellidx]
spikes_dir = paths.spikes / cell_expname


PosixPath('/mnt/z/Omer-Lab-Shared/data/analysis/SocialPlaceCells/spikes/exp_2025_01_22_001')

In [7]:
tt_token = "TT" + cell_TT_str
ntt_candidates = [
    path
    for path in sorted(spikes_dir.iterdir())
    if path.is_file() and path.suffix.lower() == ".ntt" and tt_token in path.name and "SORTED" in path.name.upper()
]
ntt_file_path = ntt_candidates[0] if ntt_candidates else None

if ntt_file_path is None:
    raise FileNotFoundError(f"No sorted .ntt file found for {tt_token} in {spikes_dir}")

result = load_ntt_clusters(ntt_file_path, 2)

cluster_2_timestamps = result[0]["timestamps_s"]
cluster_2_waveforms = result[0]["waveforms"]

In [15]:
# correct the timestamps by loading the sync paramteres from the sync file in paths.sync and applying the correction to the timestamps
sync_file = paths.sync_opt_neural /  cell_expname / "Nlg2Optitrack.mat"
sync_data = sm.loadmat(sync_file)


AttributeError: module 'statsmodels.api' has no attribute 'loadmat'

In [ ]:
# MATLAB translation:
# matching_TTL_residuals_times = TTL_timestamps_Nlg_valid - polyval(initial_polyfit_OptiTrack2Nlg, TTL_timestamps_OptiT_valid, [], muOptiTrack2Nlg1)
import h5py

sync_file = paths.sync_opt_neural / cell_expname / "Nlg2Optitrack.mat"
with h5py.File(sync_file, "r") as sync_data:
    ttl_timestamps_nlg = np.array(sync_data["TTL_timestamps_Nlg"]).squeeze()
    ttl_timestamps_optit = np.array(sync_data["TTL_timestamps_OptiT"]).squeeze()
    polyfit_nlg_to_optitrack = np.array(sync_data["polyfit_Nlg2OptiTrack"]).squeeze()
    polyfit_optitrack_to_nlg = np.array(sync_data["polyfit_OptiTrack2Nlg"]).squeeze()
    mu_nlg_to_optitrack = np.array(sync_data["muNlg2OptiTrack"]).squeeze()
    mu_optitrack_to_nlg = np.array(sync_data["muOptiTrack2Nlg"]).squeeze()

valid_ttl_mask = np.isfinite(ttl_timestamps_nlg) & np.isfinite(ttl_timestamps_optit)
ttl_timestamps_nlg_valid = ttl_timestamps_nlg[valid_ttl_mask]
ttl_timestamps_optit_valid = ttl_timestamps_optit[valid_ttl_mask]

predicted_nlg_from_optit = np.polyval(
    polyfit_optitrack_to_nlg,
    (ttl_timestamps_optit_valid - mu_optitrack_to_nlg[0]) / mu_optitrack_to_nlg[1],
)
matching_TTL_residuals_times = ttl_timestamps_nlg_valid - predicted_nlg_from_optit

cluster_2_timestamps_optitrack = np.polyval(
    polyfit_nlg_to_optitrack,
    (cluster_2_timestamps - mu_nlg_to_optitrack[0]) / mu_nlg_to_optitrack[1],
)

print(f"Mean TTL residual: {matching_TTL_residuals_times.mean():.6f} s")
print(f"Residual std: {matching_TTL_residuals_times.std():.6f} s")

cluster_2_timestamps_optitrack

In [14]:
print(cell_expname)
print(paths.sync_opt_neural)
print(paths.sync_opt_neural / cell_expname / 'Nlg2Optitrack.mat')

exp_2025_01_22_001
/mnt/z/Omer-Lab-Shared/data/analysis/SocialPlaceCells/sync
/mnt/z/Omer-Lab-Shared/data/analysis/SocialPlaceCells/sync/exp_2025_01_22_001/Nlg2Optitrack.mat
